# `gold.dim_mandate` — load

`gold.dim_ticker` → **`gold.dim_mandate`**. One row per `aic_sector` value. Expected: **32 rows**.

This is the one table in the pipeline whose values are **authored rather than derived**. The
AIC publishes the sector string; it does not publish the split into region, asset class and
style. So the mapping is a `CASE` written out in full, long and boring on purpose — it can be
read and disagreed with, which a cleverer lookup could not.

**`region = 'NotStated'` covers 37 of 102 tickers**, and that is the honest answer rather than
the tidy one. Every sector in it is classified by *what* it invests in rather than *where* —
Private Equity, Renewable Energy Infrastructure, Healthcare, Flexible Investment. Calling them
all "Global" would fill the column with an assertion the source does not support. The region
cut therefore runs over the **62 tickers with a stated geography**, and says so.

In [ ]:
CREATE OR REPLACE TEMP VIEW gold_stage_dim_mandate AS
WITH sectors AS (
  SELECT aic_sector, COUNT(DISTINCT ticker) AS trusts_in_mandate
  FROM `index-vs-trust-pipeline`.gold.dim_ticker
  WHERE is_current
  GROUP BY aic_sector
),
mapped AS (
  SELECT aic_sector, trusts_in_mandate,
         -- WHERE it invests. NotStated is a real answer, not a gap to be filled.
         CASE
           WHEN aic_sector IN ('Global', 'Global Equity Income',
                               'Global Smaller Companies')            THEN 'Global'
           WHEN aic_sector IN ('UK Equity Income', 'UK All Companies',
                               'UK Smaller Companies')                THEN 'UK'
           WHEN aic_sector IN ('Europe', 'European Smaller Companies') THEN 'Europe'
           WHEN aic_sector IN ('North America')                        THEN 'North America'
           WHEN aic_sector IN ('Japan', 'Japanese Smaller Companies')  THEN 'Japan'
           WHEN aic_sector IN ('Asia Pacific', 'Asia Pacific Equity Income',
                               'Asia Pacific Smaller Companies')      THEN 'Asia Pacific'
           WHEN aic_sector IN ('Global Emerging Markets', 'India / Indian Subcontinent',
                               'China / Greater China',
                               'Country Specialist')                  THEN 'Emerging Markets'
           WHEN aic_sector = 'NotApplicable'                          THEN 'NotApplicable'
           ELSE 'NotStated'
         END AS region,
         -- WHAT the money actually buys.
         CASE
           WHEN aic_sector IN ('Private Equity', 'Growth Capital')    THEN 'Private Equity'
           WHEN aic_sector IN ('Renewable Energy Infrastructure',
                               'Infrastructure')                      THEN 'Infrastructure'
           WHEN aic_sector IN ('Debt - Direct Lending',
                               'Debt - Structured Finance')           THEN 'Debt'
           WHEN aic_sector = 'Property Securities'                    THEN 'Property'
           WHEN aic_sector = 'Commodities & Natural Resources'        THEN 'Commodities'
           WHEN aic_sector = 'Flexible Investment'                    THEN 'Multi-asset'
           WHEN aic_sector = 'NotApplicable'                          THEN 'NotApplicable'
           ELSE 'Equity'
         END AS asset_class,
         -- HOW it is run. Income and Smaller Companies are named in the sector; the rest is Broad.
         CASE
           WHEN aic_sector = 'NotApplicable'                          THEN 'NotApplicable'
           WHEN aic_sector LIKE '%Equity Income%'
             OR aic_sector LIKE 'Debt - %'                            THEN 'Income'
           WHEN aic_sector LIKE '%Smaller Companies%'                 THEN 'Smaller Companies'
           ELSE 'Broad'
         END AS style
  FROM sectors
)
SELECT MD5(aic_sector) AS mandate_key,
       aic_sector, region, asset_class, style, trusts_in_mandate
FROM mapped;

In [ ]:
MERGE INTO `index-vs-trust-pipeline`.gold.dim_mandate AS t
USING gold_stage_dim_mandate AS s
   ON t.mandate_key = s.mandate_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
-- A sector no trust is in any more is not history worth keeping: it describes the universe,
-- and the universe is whatever dim_ticker currently holds.
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

Expected: **32 rows**, **0** duplicate keys, and `trusts_in_mandate` summing to **102** —
every ticker lands in exactly one mandate, so the fact cannot fan out through this join.

In [ ]:
SELECT COUNT(*)                                   AS rows_total,
       COUNT(*) - COUNT(DISTINCT mandate_key)     AS duplicate_keys,
       SUM(trusts_in_mandate)                     AS tickers_covered,
       COUNT(DISTINCT region)                     AS regions,
       COUNT(DISTINCT asset_class)                AS asset_classes,
       COUNT(DISTINCT style)                      AS styles
FROM `index-vs-trust-pipeline`.gold.dim_mandate;

Expect **32 / 0 / 102 / 9 / 8 / 4**.

If `tickers_covered` is not 102 the mapping has lost a sector, and every trust in it would
disappear from the fact.

In [ ]:
-- The region cut, which is the whole point of the table. Expected exactly these nine.
SELECT region,
       SUM(trusts_in_mandate) AS tickers,
       COUNT(*)               AS sectors,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(aic_sector))) AS covers
FROM `index-vs-trust-pipeline`.gold.dim_mandate
GROUP BY region
ORDER BY tickers DESC;

Expect:

| region | tickers |
|---|---|
| NotStated | **37** |
| Global | **15** |
| UK | **13** |
| Emerging Markets | **10** |
| Asia Pacific | **9** |
| Europe | **7** |
| Japan | **4** |
| North America | **4** |
| NotApplicable | **3** |

**Eight usable buckets where the raw sector gave thirty-two**, and the three index tickers
sit in their own member rather than being filtered out by name.

`NotStated` being the largest is the finding to state rather than hide: **a third of the UK
trust universe is defined by what it holds, not where it invests.**